# Inverse-dynamics weight sweep

Loads planning success for the per-environment lambda sweep and plots success
rate vs. inverse weight for all four environments. Run the `train_<env>` and
`eval_<env>` stages first; the figure is displayed inline (nothing is written to disk).

In [ ]:
# Paper figure (Fig. 7): inverse-lambda planning sweep across environments.
import json
import math
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(f"Could not find repository root from {path}")


REPO_ROOT = find_repo_root()
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".cache" / "matplotlib"))

import matplotlib.pyplot as plt
from matplotlib.ticker import FixedFormatter, FixedLocator, NullFormatter

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from plot_style import FULL_WIDTH, apply_matplotlib_style, palette

apply_matplotlib_style()

ENV_ORDER = ["TwoRoom", "Reacher", "Push-T", "OGBench-Cube"]
LAMBDA_SWEEP = [
    (0.0, "lambda_0"),
    (0.1, "lambda_0p1"),
    (0.3, "lambda_0p3"),
    (1.0, "lambda_1"),
    (3.0, "lambda_3"),
    (10.0, "lambda_10"),
    (30.0, "lambda_30"),
    (100.0, "lambda_100"),
]
ENV_CONFIG = {
    "TwoRoom": {
        "experiment_dir": REPO_ROOT / "experiments" / "lambda_sweep" / "eval_tworoom",
        "goal_tag": "goal_offset_025_fix_budget",
        "main_lambda": 0.1,
    },
    "Reacher": {
        "experiment_dir": REPO_ROOT / "experiments" / "lambda_sweep" / "eval_reacher",
        "goal_tag": "goal_offset_025",
        "main_lambda": 5.0,
    },
    "Push-T": {
        "experiment_dir": REPO_ROOT / "experiments" / "lambda_sweep" / "eval_pusht",
        "goal_tag": "goal_offset_025_fix_budget",
        "main_lambda": 30.0,
    },
    "OGBench-Cube": {
        "experiment_dir": REPO_ROOT / "experiments" / "lambda_sweep" / "eval_ogbcube",
        "goal_tag": "goal_offset_025",
        "main_lambda": 1.0,
    },
}

ZERO_LOG_POSITION = 0.015
SUCCESS_ALIASES = ("success_rate", "success", "success_mean")


def lambda_to_plot(value):
    return ZERO_LOG_POSITION if float(value) == 0.0 else float(value)


def parse_episode_successes(value):
    if isinstance(value, list):
        return [bool(item) for item in value]
    if not isinstance(value, str):
        return []
    return [token == "True" for token in re.findall(r"True|False", value)]


def success_rate_from_metrics(metrics):
    for key in SUCCESS_ALIASES:
        if key in metrics:
            value = float(metrics[key])
            return value * 100.0 if value <= 1.0 else value
    successes = parse_episode_successes(metrics.get("episode_successes"))
    if successes:
        return 100.0 * float(np.mean(successes))
    raise KeyError("No success-rate metric found")


def seed_from_payload(payload, metrics, metrics_path):
    seed_value = metrics.get("seed", metrics.get("seeds", payload.get("seed")))
    if seed_value is not None:
        return str(seed_value)
    match = re.search(r"seed[_-]?(\d+)", str(metrics_path))
    return match.group(1) if match else None


def load_lambda_sweep_rows():
    rows = []
    missing = []
    for env_name in ENV_ORDER:
        config = ENV_CONFIG[env_name]
        for lambda_value, lambda_label in LAMBDA_SWEEP:
            metrics_paths = sorted(
                (config["experiment_dir"] / "results" / lambda_label / config["goal_tag"]).glob("**/metrics.json")
            )
            if not metrics_paths:
                missing.append((env_name, lambda_value, config["goal_tag"]))
                continue
            for metrics_path in metrics_paths:
                payload = json.loads(metrics_path.read_text(encoding="utf-8"))
                metrics = payload.get("metrics", payload)
                episode_successes = parse_episode_successes(metrics.get("episode_successes"))
                rows.append({
                    "environment": env_name,
                    "lambda": float(lambda_value),
                    "lambda_label": lambda_label,
                    "goal_tag": config["goal_tag"],
                    "seed": seed_from_payload(payload, metrics, metrics_path),
                    "success_rate": success_rate_from_metrics(metrics),
                    "n_eval_episodes": len(episode_successes),
                    "episode_success_count": int(sum(episode_successes)) if episode_successes else "",
                    "source": str(metrics_path.relative_to(REPO_ROOT)),
                })
    return rows, missing


def summarize_rows(rows):
    summary = []
    for env_name in ENV_ORDER:
        for lambda_value, lambda_label in LAMBDA_SWEEP:
            group = [row for row in rows if row["environment"] == env_name and row["lambda"] == lambda_value]
            if not group:
                continue
            values = np.asarray([row["success_rate"] for row in group], dtype=float)
            episode_counts = [row["n_eval_episodes"] for row in group if row["n_eval_episodes"]]
            success_counts = [row["episode_success_count"] for row in group if row["episode_success_count"] != ""]
            mean_success = float(np.mean(values))
            std_success = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0
            if len(values) > 1:
                sem_success = float(std_success / math.sqrt(len(values)))
                uncertainty_source = "seed_sem"
            elif episode_counts and success_counts:
                n_eval = int(episode_counts[0])
                p_hat = float(success_counts[0]) / float(n_eval)
                sem_success = 100.0 * math.sqrt(p_hat * (1.0 - p_hat) / n_eval)
                uncertainty_source = "episode_binomial_sem"
            else:
                sem_success = 0.0
                uncertainty_source = "none"
            summary.append({
                "environment": env_name,
                "lambda": float(lambda_value),
                "lambda_label": lambda_label,
                "x_plot": lambda_to_plot(lambda_value),
                "mean_success_rate": mean_success,
                "std_success_rate": std_success,
                "sem_success_rate": sem_success,
                "n_runs": int(len(values)),
                "n_eval_episodes": int(episode_counts[0]) if episode_counts else "",
                "uncertainty_source": uncertainty_source,
                "sources": ";".join(row["source"] for row in group),
            })
    return summary


rows, missing = load_lambda_sweep_rows()
summary = summarize_rows(rows)

# Show the underlying numbers inline; nothing is written to disk.
display(pd.DataFrame(summary, columns=[
    "environment", "lambda", "mean_success_rate", "sem_success_rate",
    "n_runs", "n_eval_episodes", "uncertainty_source",
]))

summary_by_env = {env: [row for row in summary if row["environment"] == env] for env in ENV_ORDER}

fig, axes = plt.subplots(1, len(ENV_ORDER), figsize=(FULL_WIDTH, 1.62), sharey=True)
for ax, env_name in zip(axes, ENV_ORDER):
    env_rows = summary_by_env[env_name]
    x = np.asarray([row["x_plot"] for row in env_rows], dtype=float)
    y = np.asarray([row["mean_success_rate"] / 100.0 for row in env_rows], dtype=float)
    err = np.asarray([row["sem_success_rate"] / 100.0 for row in env_rows], dtype=float)

    order = np.argsort(x)
    x, y, err = x[order], y[order], err[order]
    ax.fill_between(
        x,
        np.clip(y - err, 0.0, 1.0),
        np.clip(y + err, 0.0, 1.0),
        color=palette["Med Blue"],
        alpha=0.22,
        linewidth=0,
    )
    ax.plot(
        x,
        y,
        color=palette["Dark Blue"],
        marker="o",
        markersize=3.2,
        markerfacecolor="white",
        markeredgewidth=0.75,
        linewidth=1.25,
    )

    main_lambda = ENV_CONFIG[env_name]["main_lambda"]
    ax.axvline(lambda_to_plot(main_lambda), color=palette["Dark Red"], linestyle=":", linewidth=1.0, alpha=0.9)

    ax.set_title(env_name, pad=3)
    ax.set_xscale("log")
    ax.set_xlim(0.012, 140)
    ax.set_ylim(0.0, 1.04)
    ax.xaxis.set_major_locator(FixedLocator([ZERO_LOG_POSITION, 0.1, 1.0, 10.0, 100.0]))
    ax.xaxis.set_major_formatter(FixedFormatter(["0", "0.1", "1", "10", "100"]))
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.set_yticks([0.0, 0.5, 1.0])
    ax.set_yticklabels(["0", "0.5", "1.0"])
    ax.grid(axis="y", linewidth=0.45)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="x", pad=1.5)
    if ax is not axes[0]:
        ax.tick_params(axis="y", length=0)

axes[0].set_ylabel("Success rate")
fig.supxlabel("Inverse weight $\\lambda$", y=0.02, fontsize=plt.rcParams["axes.labelsize"])
fig.subplots_adjust(left=0.075, right=0.985, bottom=0.30, top=0.84, wspace=0.18)
plt.show()

print(f"Loaded {len(rows)} result files for {len(summary)} environment/lambda pairs.")
if missing:
    print("Missing environment/lambda result files:")
    for env_name, lambda_value, goal_tag in missing:
        print(f"  {env_name} lambda={lambda_value:g} goal_tag={goal_tag}")